####RFE and Mutual Information to choose optimal variable for the dataset (and study potentially redundant ones)

In [8]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("../laptop_data_cleaned.csv")

# handle these better with your own dataset, here we are simply dropping these
#df = df.drop(['id', 'date', 'zipcode', 'lat', 'long'], axis=1)

# change this to your own target variable
TARGET_VARIABLE = "Price"

In [9]:
####RFE - recursive feature elimination

In [10]:
# typical X/y -split
X = df.drop(TARGET_VARIABLE, axis=1)
y = df[TARGET_VARIABLE]

# encode text columns so every feature is numeric
X = pd.get_dummies(X, drop_first=True).astype(float)

model = RandomForestRegressor(random_state=42)

rfe = RFE(estimator=model, n_features_to_select=7)
rfe.fit(X, y)

rankings = rfe.ranking_
support = rfe.support_

results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

results_df

,Feature,Ranking,Selected
0,Ram,1,True
1,Weight,1,True
6,SSD,1,True
4,Ppi,1,True
27,TypeName_Notebook,1,True
31,Cpu_brand_Intel Core i5,1,True
33,Cpu_brand_Other Intel Processor,1,True
32,Cpu_brand_Intel Core i7,2,False
36,Os_Others,3,False
29,TypeName_Workstation,4,False


In [12]:
# include the categories too, via one-hot encoding
df_encoded = pd.get_dummies(df, drop_first=True).astype(float)
df_encoded.corr()[TARGET_VARIABLE].sort_values(ascending=False)

Price                              1.000000
Ram                                0.680519
SSD                                0.658808
Cpu_brand_Intel Core i7            0.593245
Ppi                                0.480687
TypeName_Gaming                    0.362537
Gpu_brand_Nvidia                   0.357157
TypeName_Ultrabook                 0.306406
Os_Windows                         0.294626
Ips                                0.279240
TypeName_Workstation               0.206745
TouchScreen                        0.194289
Company_MSI                        0.188654
Weight                             0.151386
Company_Razer                      0.133549
Company_Apple                      0.093356
Company_Dell                       0.084610
Company_Toshiba                    0.069182
Company_LG                         0.062273
Company_Microsoft                  0.052978
Company_Samsung                    0.044365
Company_Google                     0.042908
Company_Huawei                  

####Mutual information - another alternative

In [13]:
# fit the mutual information algorithm
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

Ram                                0.491231
Ppi                                0.404462
SSD                                0.370587
Weight                             0.287433
TypeName_Notebook                  0.224826
Cpu_brand_Intel Core i7            0.221797
Cpu_brand_Other Intel Processor    0.159460
Cpu_brand_Intel Core i3            0.126000
Cpu_brand_Intel Core i5            0.125478
Gpu_brand_Nvidia                   0.087836
TypeName_Gaming                    0.079505
TypeName_Ultrabook                 0.078211
HDD                                0.073901
Os_Others                          0.071166
Os_Windows                         0.061256
Gpu_brand_Intel                    0.039906
Ips                                0.031378
TouchScreen                        0.030194
TypeName_Workstation               0.027990
TypeName_Netbook                   0.021355
Company_MSI                        0.017816
Company_Mediacom                   0.014030
Company_Vero                    